<img src="../assets/logo.png"/> <br>
# A full business solution our first project

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AIz') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=api_key
)
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

API key looks good so far


In [3]:
links = fetch_website_links("https://huggingface.com")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/nvidia/LocateAnything-3B',
 '/google/gemma-4-12B-it',
 '/unsloth/gemma-4-12b-it-GGUF',
 '/google/gemma-4-12B',
 '/ideogram-ai/ideogram-4-fp8',
 '/models',
 '/spaces/VAST-AI/TripoSplat',
 '/spaces/ideogram-ai/ideogram4',
 '/spaces/nvidia/LocateAnything',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview-2',
 '/spaces/webml-community/bonsai-image-webgpu',
 '/spaces',
 '/datasets/wikimedia/structured-wikipedia',
 '/datasets/stanford-vision-lab/gpic',
 '/datasets/openbmb/UltraData-SFT-2605',
 '/datasets/nvidia/Nemotron-Personas-El-Salvador',
 '/datasets/angryg

## First step: Have gemini-2.5-flash figure out which links are relevant

### Use a call to gemini-2.5-flash to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [10]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [11]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://huggingface.com"))


Here is the list of links on the website https://huggingface.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/nvidia/LocateAnything-3B
/google/gemma-4-12B-it
/unsloth/gemma-4-12b-it-GGUF
/google/gemma-4-12B
/ideogram-ai/ideogram-4-fp8
/models
/spaces/VAST-AI/TripoSplat
/spaces/ideogram-ai/ideogram4
/spaces/nvidia/LocateAnything
/spaces/r3gm/wan2-2-fp8da-aoti-preview-2
/spaces/webml-community/bonsai-image-webgpu
/spaces
/datasets/wikimedia/structured-wikipedia
/datasets/stanford-visi

In [5]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://huggingface.com")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.com/'},
  {'type': 'models platform', 'url': 'https://huggingface.com/models'},
  {'type': 'datasets platform', 'url': 'https://huggingface.com/datasets'},
  {'type': 'spaces platform', 'url': 'https://huggingface.com/spaces'},
  {'type': 'documentation', 'url': 'https://huggingface.com/docs'},
  {'type': 'enterprise solutions',
   'url': 'https://huggingface.com/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.com/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.com/blog'},
  {'type': 'learning resources', 'url': 'https://huggingface.com/learn'},
  {'type': 'professional services', 'url': 'https://huggingface.com/pro'},
  {'type': 'support page', 'url': 'https://huggingface.com/support'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 

In [38]:
MODEL="llama3.2"
import time
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    time.sleep(20)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://huggingface.com")

## Second step: make the brochure!

Assemble all the details into another prompt to gemini-2.5-flash

In [1]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [41]:
print(fetch_page_and_all_relevant_links("https://github.com"))

Selecting relevant links for https://github.com by calling llama3.2
Found 5 relevant links
## Landing Page:

GitHub · Change is constant. GitHub keeps you ahead. · GitHub

Skip to content
Navigation Menu
Toggle navigation
Sign in
Platform
AI CODE CREATION
GitHub Copilot
Write better code with AI
GitHub Copilot app
Direct agents from issue to merge
MCP Registry
New
Integrate external tools
DEVELOPER WORKFLOWS
Actions
Automate any workflow
Codespaces
Instant dev environments
Issues
Plan and track work
Code Review
Manage code changes
APPLICATION SECURITY
GitHub Advanced Security
Find and fix vulnerabilities
Code security
Secure your code as you build
Secret protection
Stop leaks before they start
EXPLORE
Why GitHub
Documentation
Blog
Changelog
Marketplace
View all features
Solutions
BY COMPANY SIZE
Enterprises
Small and medium teams
Startups
Nonprofits
BY USE CASE
App Modernization
DevSecOps
DevOps
CI/CD
View all use cases
BY INDUSTRY
Healthcare
Financial services
Manufacturing
Government

In [3]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [4]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [44]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.com")

Selecting relevant links for https://huggingface.com by calling llama3.2
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\nabout 18 hours ago\n•\n124k\n•\n1.68k\ngoogle/gemma-4-12B-it\nUpdated\n5 days ago\n•\n581k\

In [45]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.com")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [2]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [13]:
stream_brochure("Github", "https://github.com")

# About GitHub
Join the world's largest community of developer collaboration and innovation.

At GitHub, we're passionate about empowering developers to build and maintain everything from simple personal projects to complex global services. Our platform is designed to make it easy for anyone to write, manage, and collaborate on code.

## Company Culture

Our culture is built around a few core values:

* **Collaboration**: We believe that the best ideas come from diverse perspectives and collaborative environments.
* **Openness**: We're committed to open-source principles and transparency in all aspects of our business.
* **Innovation**: We encourage experimentation, risk-taking, and continuous learning.

Our team consists of talented individuals from around the world who share these values. We offer a range of benefits and perks, including flexible work arrangements, comprehensive health insurance, and access to cutting-edge tech gear.

## Customers

We serve a diverse range of customers across various industries, including:

* **Enterprise**: Companies like IBM, Dell, and Intel rely on GitHub for their development needs.
* **Startups**: We're a popular choice among startups like Airbnb, Dropbox, and Slack.
* **Education**: We offer resources and tools to support educators and students.

## Developers

We're home to over 40 million registered users, covering industries such as:

* **Healthcare**
* **Financial Services**
* **Manufacturing**
* **Government**

Developers trust us for our platform's reliability, scalability, and innovative features. Our platform offers:

* **Automation**: Use Actions to automate workflows and tasks.
* **DevOps**: Leverage Codespaces and Automate any workflow with ease.
* **Security**: Employ advanced security features like GitHub Advanced Security.
* **AI**: Introduce GitHub Copilot, an AI-powered developer tool.

## Careers
Join our team of talented developers, designers, and engineers and contribute to shaping the future of software development.

Stay up-to-date on company news, blog posts, and announcements:

* Follow us on Twitter: <https://twitter.com/AboutGitHub>
* Visit our blog: [LinkedIn](https://www.linkedin.com/company/gitHub/)
* Sign up for our email newsletter: Contact us here
* Address Feedback:
[<initiative@gitHub.org"](mailto:initiative@gitHub.org)


In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype.</span>
        </td>
    </tr>
</table>